# Lab | Deploying a Data Science Project with Django
## Option A: Iris Classification Model

This notebook:
1. Trains and saves the Iris classification model as `model.pkl`.
2. Scaffolds the Django project (`ml_project`) and app (`predictor`).
3. Writes all required files: settings, URLs, views, templates.
4. Instructs how to run the Django server and take the required screenshot.

In [1]:
# Install all required packages
# !pip install django scikit-learn pandas numpy joblib

In [2]:
import os
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Root directory of the lab project
PROJECT_ROOT = Path.cwd() / "ml_lab_django"
PROJECT_ROOT.mkdir(exist_ok=True)
print("Project root:", PROJECT_ROOT)

Project root: d:\edu_laptop\Ironhack_AI_DataScience\Months3-5\week39\Monday\lab-django-ds-deployment\ml_lab_django


In [3]:
# Load Iris dataset
iris = load_iris()
X, y = iris.data, iris.target

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train a Random Forest classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Save model to project root as model.pkl
MODEL_PATH = PROJECT_ROOT / "model.pkl"
joblib.dump(model, MODEL_PATH)
print("Model saved to:", MODEL_PATH)

Test accuracy: 0.9000
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30

Model saved to: d:\edu_laptop\Ironhack_AI_DataScience\Months3-5\week39\Monday\lab-django-ds-deployment\ml_lab_django\model.pkl


In [4]:
# We create the Django project and app folders manually so we control every file.
DJANGO_PROJECT = PROJECT_ROOT / "ml_project"
DJANGO_APP = DJANGO_PROJECT / "predictor"
TEMPLATES_DIR = DJANGO_APP / "templates" / "predictor"

for p in [DJANGO_PROJECT, DJANGO_APP, TEMPLATES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Created:")
print(DJANGO_PROJECT)
print(DJANGO_APP)
print(TEMPLATES_DIR)

Created:
d:\edu_laptop\Ironhack_AI_DataScience\Months3-5\week39\Monday\lab-django-ds-deployment\ml_lab_django\ml_project
d:\edu_laptop\Ironhack_AI_DataScience\Months3-5\week39\Monday\lab-django-ds-deployment\ml_lab_django\ml_project\predictor
d:\edu_laptop\Ironhack_AI_DataScience\Months3-5\week39\Monday\lab-django-ds-deployment\ml_lab_django\ml_project\predictor\templates\predictor


In [5]:
manage_py = '''#!/usr/bin/env python
"""Django's command-line utility for administrative tasks."""
import os
import sys

def main():
    os.environ.setdefault("DJANGO_SETTINGS_MODULE", "ml_project.settings")
    try:
        from django.core.management import execute_from_command_line
    except ImportError as exc:
        raise ImportError(
            "Couldn't import Django. Are you sure it's installed and "
            "available on your PYTHONPATH environment variable? Did you "
            "forget to activate a virtual environment?"
        ) from exc
    execute_from_command_line(sys.argv)

if __name__ == "__main__":
    main()
'''

(DJANGO_PROJECT / "manage.py").write_text(manage_py)
print("Wrote manage.py")

Wrote manage.py


In [6]:
(DJANGO_PROJECT / "__init__.py").write_text("")

settings_py = '''"""
Django settings for ml_project.
"""
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent

SECRET_KEY = "django-insecure-replace-me-for-lab-only"
DEBUG = True
ALLOWED_HOSTS = ["*"]

INSTALLED_APPS = [
    "django.contrib.admin",
    "django.contrib.auth",
    "django.contrib.contenttypes",
    "django.contrib.sessions",
    "django.contrib.messages",
    "django.contrib.staticfiles",
    "predictor",   # our app
]

MIDDLEWARE = [
    "django.middleware.security.SecurityMiddleware",
    "django.contrib.sessions.middleware.SessionMiddleware",
    "django.middleware.common.CommonMiddleware",
    "django.middleware.csrf.CsrfViewMiddleware",
    "django.contrib.auth.middleware.AuthenticationMiddleware",
    "django.contrib.messages.middleware.MessageMiddleware",
    "django.middleware.clickjacking.XFrameOptionsMiddleware",
]

ROOT_URLCONF = "ml_project.urls"

TEMPLATES = [
    {
        "BACKEND": "django.template.backends.django.DjangoTemplates",
        "DIRS": [],
        "APP_DIRS": True,
        "OPTIONS": {
            "context_processors": [
                "django.template.context_processors.debug",
                "django.template.context_processors.request",
                "django.contrib.auth.context_processors.auth",
                "django.contrib.messages.context_processors.messages",
            ],
        },
    },
]

WSGI_APPLICATION = "ml_project.wsgi.application"

DATABASES = {
    "default": {
        "ENGINE": "django.db.backends.sqlite3",
        "NAME": BASE_DIR / "db.sqlite3",
    }
}

AUTH_PASSWORD_VALIDATORS = []

LANGUAGE_CODE = "en-us"
TIME_ZONE = "UTC"
USE_I18N = True
USE_TZ = True

STATIC_URL = "static/"

DEFAULT_AUTO_FIELD = "django.db.models.BigAutoField"

# Path to the trained model (model.pkl lives in the project root)
MODEL_PATH = BASE_DIR / "model.pkl"
'''

(DJANGO_PROJECT / "settings.py").write_text(settings_py)
print("Wrote settings.py")

Wrote settings.py


In [7]:
urls_py = '''from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path("admin/", admin.site.urls),
    path("", include("predictor.urls")),
]
'''
(DJANGO_PROJECT / "urls.py").write_text(urls_py)

wsgi_py = '''import os
from django.core.wsgi import get_wsgi_application

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "ml_project.settings")
application = get_wsgi_application()
'''
(DJANGO_PROJECT / "wsgi.py").write_text(wsgi_py)

print("Wrote urls.py and wsgi.py")

Wrote urls.py and wsgi.py


In [8]:
(DJANGO_APP / "__init__.py").write_text("")

apps_py = '''from django.apps import AppConfig

class PredictorConfig(AppConfig):
    default_auto_field = "django.db.models.BigAutoField"
    name = "predictor"
'''
(DJANGO_APP / "apps.py").write_text(apps_py)

(DJANGO_APP / "models.py").write_text("# No database models required for this lab.\n")
(DJANGO_APP / "admin.py").write_text("from django.contrib import admin\n")

print("Wrote app core files")

Wrote app core files


In [9]:
views_py = '''import os
import joblib
import numpy as np
from django.conf import settings
from django.shortcuts import render

# Load the trained model once at import time
MODEL = joblib.load(settings.MODEL_PATH)

# Iris metadata (must match training order)
FEATURE_NAMES = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)",
]
CLASS_NAMES = ["setosa", "versicolor", "virginica"]


def home(request):
    """Display the input form."""
    return render(request, "predictor/home.html", {"feature_names": FEATURE_NAMES})


def predict(request):
    """Handle form submission and return a prediction."""
    if request.method != "POST":
        return render(request, "predictor/home.html", {"feature_names": FEATURE_NAMES})

    errors = []
    values = []

    for name in FEATURE_NAMES:
        raw = request.POST.get(name, "").strip()
        if raw == "":
            errors.append(f"Missing value for {name}.")
            continue
        try:
            values.append(float(raw))
        except ValueError:
            errors.append(f"Invalid number for {name}: '{raw}'.")

    if errors:
        return render(
            request,
            "predictor/home.html",
            {"feature_names": FEATURE_NAMES, "errors": errors, "posted": request.POST},
        )

    X = np.array(values).reshape(1, -1)
    pred_idx = int(MODEL.predict(X)[0])
    proba = MODEL.predict_proba(X)[0]
    pred_label = CLASS_NAMES[pred_idx]

    context = {
        "feature_names": FEATURE_NAMES,
        "inputs": dict(zip(FEATURE_NAMES, values)),
        "prediction": pred_label,
        "probabilities": [
            {"class": CLASS_NAMES[i], "prob": round(float(proba[i]) * 100, 2)}
            for i in range(len(CLASS_NAMES))
        ],
    }
    return render(request, "predictor/result.html", context)
'''
(DJANGO_APP / "views.py").write_text(views_py)
print("Wrote views.py")

Wrote views.py


In [10]:
app_urls_py = '''from django.urls import path
from . import views

urlpatterns = [
    path("", views.home, name="home"),
    path("predict/", views.predict, name="predict"),
]
'''
(DJANGO_APP / "urls.py").write_text(app_urls_py)
print("Wrote predictor/urls.py")

Wrote predictor/urls.py


In [11]:
base_html = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>{% block title %}Iris Predictor{% endblock %}</title>
  <link
    href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css"
    rel="stylesheet">
</head>
<body class="bg-light">
  <nav class="navbar navbar-expand-lg navbar-dark bg-primary mb-4">
    <div class="container">
      <a class="navbar-brand" href="{% url 'home' %}">Iris Predictor</a>
    </div>
  </nav>

  <main class="container">
    {% block content %}{% endblock %}
  </main>

  <footer class="text-center text-muted mt-5 mb-3">
    <small>Django + scikit-learn · Iris classification lab</small>
  </footer>
</body>
</html>
'''
(TEMPLATES_DIR / "base.html").write_text(base_html)
print("Wrote base.html")

Wrote base.html


In [12]:
home_html = '''{% extends "predictor/base.html" %}
{% block title %}Iris Predictor — Home{% endblock %}

{% block content %}
<div class="row justify-content-center">
  <div class="col-md-8">
    <div class="card shadow-sm">
      <div class="card-body">
        <h3 class="card-title mb-3">Enter Iris measurements</h3>

        {% if errors %}
          <div class="alert alert-danger">
            <ul class="mb-0">
              {% for e in errors %}<li>{{ e }}</li>{% endfor %}
            </ul>
          </div>
        {% endif %}

        <form method="post" action="{% url 'predict' %}">
          {% csrf_token %}
          {% for name in feature_names %}
            <div class="mb-3">
              <label class="form-label" for="f{{ forloop.counter }}">{{ name }}</label>
              <input
                type="number"
                step="any"
                class="form-control"
                id="f{{ forloop.counter }}"
                name="{{ name }}"
                value="{{ posted|default_if_none:''|get_item:name }}"
                required>
            </div>
          {% endfor %}
          <button type="submit" class="btn btn-primary">Predict</button>
        </form>
      </div>
    </div>
  </div>
</div>
{% endblock %}
'''
(TEMPLATES_DIR / "home.html").write_text(home_html)
print("Wrote home.html")

Wrote home.html


In [13]:
home_html = '''{% extends "predictor/base.html" %}
{% block title %}Iris Predictor — Home{% endblock %}

{% block content %}
<div class="row justify-content-center">
  <div class="col-md-8">
    <div class="card shadow-sm">
      <div class="card-body">
        <h3 class="card-title mb-3">Enter Iris measurements</h3>

        {% if errors %}
          <div class="alert alert-danger">
            <ul class="mb-0">
              {% for e in errors %}<li>{{ e }}</li>{% endfor %}
            </ul>
          </div>
        {% endif %}

        <form method="post" action="{% url 'predict' %}">
          {% csrf_token %}
          {% for name in feature_names %}
            <div class="mb-3">
              <label class="form-label" for="f{{ forloop.counter }}">{{ name }}</label>
              <input
                type="number"
                step="any"
                class="form-control"
                id="f{{ forloop.counter }}"
                name="{{ name }}"
                required>
            </div>
          {% endfor %}
          <button type="submit" class="btn btn-primary">Predict</button>
        </form>
      </div>
    </div>
  </div>
</div>
{% endblock %}
'''
(TEMPLATES_DIR / "home.html").write_text(home_html)
print("Wrote simplified home.html")

Wrote simplified home.html


In [14]:
result_html = '''{% extends "predictor/base.html" %}
{% block title %}Iris Predictor — Result{% endblock %}

{% block content %}
<div class="row justify-content-center">
  <div class="col-md-8">
    <div class="card shadow-sm">
      <div class="card-body">
        <h3 class="card-title mb-3">Prediction Result</h3>

        <p class="fs-4">
          Predicted class:
          <span class="badge bg-success">{{ prediction }}</span>
        </p>

        <h5 class="mt-4">Input values</h5>
        <ul class="list-group mb-4">
          {% for name, value in inputs.items %}
            <li class="list-group-item d-flex justify-content-between">
              <span>{{ name }}</span>
              <strong>{{ value }}</strong>
            </li>
          {% endfor %}
        </ul>

        <h5>Class probabilities</h5>
        <ul class="list-group mb-4">
          {% for p in probabilities %}
            <li class="list-group-item d-flex justify-content-between">
              <span>{{ p.class }}</span>
              <strong>{{ p.prob }}%</strong>
            </li>
          {% endfor %}
        </ul>

        <a href="{% url 'home' %}" class="btn btn-outline-primary">Try another</a>
      </div>
    </div>
  </div>
</div>
{% endblock %}
'''
(TEMPLATES_DIR / "result.html").write_text(result_html)
print("Wrote result.html")

Wrote result.html


In [15]:
# The Django project expects model.pkl next to manage.py
import shutil
dst = DJANGO_PROJECT / "model.pkl"
shutil.copy(MODEL_PATH, dst)
print("Model copied to:", dst)
print("Exists:", dst.exists())

Model copied to: d:\edu_laptop\Ironhack_AI_DataScience\Months3-5\week39\Monday\lab-django-ds-deployment\ml_lab_django\ml_project\model.pkl
Exists: True


In [16]:
for root, dirs, files in os.walk(PROJECT_ROOT):
    # Skip __pycache__
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = len(Path(root).relative_to(PROJECT_ROOT).parts)
    indent = "  " * level
    print(f"{indent}{Path(root).name}/")
    for f in files:
        print(f"{indent}  {f}")

ml_lab_django/
  model.pkl
  ml_project/
    manage.py
    model.pkl
    settings.py
    urls.py
    wsgi.py
    __init__.py
    predictor/
      admin.py
      apps.py
      models.py
      urls.py
      views.py
      __init__.py
      templates/
        predictor/
          base.html
          home.html
          result.html


## 🚀 Run the Django server

Open a **terminal** (not the notebook), and run:

```bash
cd ml_lab_django/ml_project
python manage.py migrate
python manage.py runserver